# 10 - GCS 2024: Zonal Statistics and Cross-Validation Against GAEZ

Continues notebook 09. Runs the zonal statistics for the 6 Banten-clipped Global Crop Suitability 2024 rasters with
the helper script `_gcs2024_zonal_stats.py` as a separate process, then merges the result into
`tabular_features.csv` and cross-validates it against GAEZ with plain pandas.

Why a separate process: it was introduced when the same `zonal_stats` calls kept killing the Jupyter kernel. The real
cause was a kernel started without the conda environment activated (its `Library/bin` DLL folders and `GDAL_DATA` /
`PROJ_LIB` missing), not GDAL or ipykernel. In an activated environment, as PyCharm runs it, the calls also work as
notebook code; the subprocess gives identical results and is kept.

In [1]:
import os, sys
from pathlib import Path

NOTEBOOK_DIR = Path(os.path.abspath("."))
if NOTEBOOK_DIR.name != "preprocessing":
    NOTEBOOK_DIR = NOTEBOOK_DIR / "scripts" / "preprocessing"
os.chdir(NOTEBOOK_DIR)

PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
DOWNLOAD_DIR = PROJECT_ROOT / "scripts" / "download"
if str(DOWNLOAD_DIR) not in sys.path:
    sys.path.insert(0, str(DOWNLOAD_DIR))

from common import PROCESSED_DIR

GCS_CROPS = ["rice", "maize", "cassava", "soybean", "oilpalm", "sugarcane"]
print("cwd:", os.getcwd())

cwd: D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\scripts\preprocessing


## 1. Zonal statistics, run as a subprocess (see markdown above for why)

In [2]:
import subprocess

GCS_STATS_PATH = PROCESSED_DIR / "features" / "gcs2024_zonal_stats.csv"

result = subprocess.run(
    [sys.executable, str(NOTEBOOK_DIR / "_gcs2024_zonal_stats.py")],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f"_gcs2024_zonal_stats.py failed with exit code {result.returncode}")
assert GCS_STATS_PATH.exists(), "zonal stats output not found"


9333 grid cells loaded
rice: 9219/9333 cells got a value
maize: 9219/9333 cells got a value
cassava: 9219/9333 cells got a value
soybean: 9219/9333 cells got a value
oilpalm: 9219/9333 cells got a value
sugarcane: 9219/9333 cells got a value
Wrote D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\datasets\processed\features\gcs2024_zonal_stats.csv



## 2. Merge `gcs2024_<crop>` columns into `tabular_features.csv` (comparison only, not used for labeling)

In [3]:
import pandas as pd

FEATURES_PATH = PROCESSED_DIR / "features" / "tabular_features.csv"
df = pd.read_csv(FEATURES_PATH)
gcs_stats = pd.read_csv(GCS_STATS_PATH)

new_cols = [c for c in gcs_stats.columns if c != "grid_id"]
already_present = [c for c in new_cols if c in df.columns]
if already_present:
    print("Already present, skipping merge:", already_present)
else:
    df = df.merge(gcs_stats, on="grid_id", how="left")
    df.to_csv(FEATURES_PATH, index=False)
    print(f"Wrote {FEATURES_PATH} - {len(df)} rows, {len(df.columns)} columns")
    print("Added this run:", new_cols)

Already present, skipping merge: ['gcs2024_rice', 'gcs2024_maize', 'gcs2024_cassava', 'gcs2024_soybean', 'gcs2024_oilpalm', 'gcs2024_sugarcane']


## 3. Cross-validation: does GAEZ agree with Global Crop Suitability 2024?

In [4]:
# Spearman correlation per crop - rank correlation is more appropriate here than Pearson,
# since the two sources use different, not-directly-comparable suitability scales.
#
# Computed from the rank-difference formula rho = 1 - 6*sum(d^2) / (n*(n^2-1)); on this data it matches
# scipy.stats.spearmanr to three decimals. It was written by hand when spearmanr kept killing the process,
# which turned out to be the non-activated conda environment (missing DLL folders), not scipy: in an activated
# environment scipy.stats.spearmanr works and gives the same values.
print("Spearman correlation (manual rank-difference formula): GAEZ index vs Global Crop Suitability 2024, per crop")
for crop in GCS_CROPS:
    gaez_col, gcs_col = f"gaez_index_{crop}", f"gcs2024_{crop}"
    sub = df[[gaez_col, gcs_col]].dropna()
    r1, r2 = sub[gaez_col].rank(), sub[gcs_col].rank()
    n = len(sub)
    d2 = ((r1 - r2) ** 2).sum()
    rho = 1 - (6 * d2) / (n * (n**2 - 1))
    print(f"  {crop}: rho={rho:.3f} (n={n})")

print()
print("GAEZ's top pick among these 6 crops, by the rank that same crop holds in GCS 2024:")
import numpy as np

gaez_cols = [f"gaez_index_{c}" for c in GCS_CROPS]
gcs_cols = [f"gcs2024_{c}" for c in GCS_CROPS]
gaez_arr = df[gaez_cols].to_numpy()
gcs_arr = df[gcs_cols].to_numpy()
gaez_best_i = np.argmax(gaez_arr, axis=1)
gcs_rank_arr = (-gcs_arr).argsort(axis=1).argsort(axis=1) + 1  # 1 = highest GCS score
rank_of_gaez_pick = gcs_rank_arr[np.arange(len(df)), gaez_best_i]
print(pd.Series(rank_of_gaez_pick).value_counts().sort_index())
print("(rank 1 = GAEZ's pick is ALSO GCS 2024's top pick for that cell; higher = more disagreement)")

Spearman correlation (manual rank-difference formula): GAEZ index vs Global Crop Suitability 2024, per crop
  rice: rho=0.781 (n=9219)
  maize: rho=0.435 (n=9219)
  cassava: rho=0.359 (n=9219)
  soybean: rho=-0.174 (n=9219)
  oilpalm: rho=0.591 (n=9219)
  sugarcane: rho=0.796 (n=9219)

GAEZ's top pick among these 6 crops, by the rank that same crop holds in GCS 2024:
1    2026
2    2167
3    2177
4    2084
5     614
6     265
Name: count, dtype: int64
(rank 1 = GAEZ's pick is ALSO GCS 2024's top pick for that cell; higher = more disagreement)


## Notes

- `gcs2024_<crop>` columns are raw Global Crop Suitability 2024 scores (own 0-1 scale, not
  0-10000 like GAEZ) - kept separate from `gaez_index_<crop>`, not merged into the labeling rule.
- Correlation above shows how much the two independent models agree per crop. Low/negative
  correlation for a crop is itself a finding worth reporting, not an error to fix.
- Does **not** touch Assignment I's written report, slide deck, or presentation script.